# Week 7 capstone: Step 3, GPU smoke test and determinism-policy timing

**Scope of this notebook: smoke stage only.** It contains no calibration, pilot, or main-experiment cells, and the repository refuses to run those stages until the policy decision, the environment lock, and (for main runs) the frozen pre-registration exist. All experiment logic lives in the repository. This notebook only clones it and calls its scripts, so cell order cannot change what the experiment does.

**This notebook never runs `pip install` or upgrades anything.** It records the packages Colab provides and checks them against `requirements.lock.txt`.

Run the cells top to bottom, on a **T4 GPU runtime** (Runtime > Change runtime type). Then download the zip from the last cell and send it back for review. Do not start calibration until the outputs have been reviewed.

In [1]:
# 1. Get the code. Edit REPO_URL if yours differs. Optionally pin COMMIT to an exact commit hash.
REPO_URL = "https://github.com/sweiss1367/wk5_architecture-variants.git"
BRANCH = "claude/trusting-pascal-1qjsp0"
COMMIT = ""   # e.g. "d80ca1d..." ; leave empty to use the branch head (the exact hash is printed and recorded either way)

import os, subprocess
if not os.path.isdir("repo"):
    url = REPO_URL
    try:                                   # private repo: add a Colab secret named GITHUB_TOKEN (never printed)
        from google.colab import userdata
        tok = userdata.get("GITHUB_TOKEN")
        if tok:
            url = REPO_URL.replace("https://", f"https://{tok}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "--branch", BRANCH, url, "repo"], check=True)
    subprocess.run(["git", "-C", "repo", "remote", "set-url", "origin", REPO_URL], check=True)   # drop the token from the remote
if COMMIT:
    subprocess.run(["git", "-C", "repo", "checkout", COMMIT], check=True)
%cd repo/week7_capstone
!git rev-parse HEAD
!git status --short

/content/repo/week7_capstone
0541b40a7ba40cdf55fe4faa1749462ac1f0f7ce


In [2]:
# 2. GPU check. The study is defined for a Colab GPU; stop here if there is none.
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU: switch the runtime to a T4 GPU and restart."
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))

Mon Oct 19 17:52:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# 3. Environment: capture the lock on the first session, verify against it afterwards. Never installs or upgrades.
!python src/check_env.py --auto

pinned packages: {'torch': '2.11.0+cu130', 'numpy': '2.1.3', 'PyYAML': '6.0.3'} | python 3.13.15
lock written to requirements.lock.txt. Commit it. Later sessions are checked against it.


In [4]:
# 4. Determinism policy timing: P1 vs P2 on identical workloads, timing only (no accuracy is computed).
#    It prints the measured difference first, then applies the pre-declared feasibility rule.
!python src/choose_policy.py

warm-up run (discarded) ...
  rep 0 A S=200 P1: train 9.0s wall 15.5s
  rep 0 A S=200 P2: train 6.8s wall 12.1s
  rep 0 A S=600 P1: train 27.3s wall 32.8s
  rep 0 A S=600 P2: train 20.9s wall 26.3s
  rep 0 B S=200 P1: train 10.6s wall 16.2s
  rep 0 B S=200 P2: train 7.7s wall 13.0s
  rep 0 B S=600 P1: train 30.3s wall 35.8s
  rep 0 B S=600 P2: train 25.1s wall 30.5s
  rep 0 C S=200 P1: train 13.7s wall 19.7s
  rep 0 C S=200 P2: train 10.0s wall 15.5s
  rep 0 C S=600 P1: train 36.6s wall 42.5s
  rep 0 C S=600 P2: train 30.5s wall 36.3s
  rep 1 A S=200 P2: train 8.4s wall 14.3s
  rep 1 A S=200 P1: train 11.8s wall 18.1s
  rep 1 A S=600 P2: train 25.4s wall 30.8s
  rep 1 A S=600 P1: train 34.9s wall 40.8s
  rep 1 B S=200 P2: train 8.1s wall 13.7s
  rep 1 B S=200 P1: train 11.9s wall 18.0s
  rep 1 B S=600 P2: train 25.5s wall 31.0s
  rep 1 B S=600 P1: train 35.2s wall 41.1s
  rep 1 C S=200 P2: train 11.4s wall 17.1s
  rep 1 C S=200 P1: train 15.4s wall 21.3s
  rep 1 C S=600 P2: train 33.2s

In [9]:
# 5. Smoke checks (V4 determinism, V5 shared initialization, V6 scale, V7 demand floor, V16 memory) and the unit tests.
#    Run after step 4 so no accuracy figure can influence the policy decision.
!python src/smoke_checks.py

check                                                     status 
V4_P1_same_seed_rerun                                     PASS   (informational)
V4_P2_same_seed_rerun                                     INFO   (informational)
V5_shared_init_identical_across_arms_and_processes        PASS   
V6_positional_vs_value_embedding_scale                    PASS   
V7_demand_floor_keeps_mape_stable                         PASS   
V16_inference_memory_excludes_training_state              FAIL   
leakage_checks_inside_every_run                           PASS   
unit_tests                                                PASS   


In [10]:
# 6. Summary of what was recorded.
import json
d = json.load(open("results/determinism_decision.json"))
print("selected policy:", d["selected_policy"], "| accuracy used:", d["accuracy_used"], "| feasible:", d["feasible_under_selected"])
print("projected total hours:", d["projected_total_hours"], "| ceiling:", d["ceiling_hours"])
print("measured penalty % (train step, P1 vs P2):", d["measured_penalty_percent_train_step"])
print("GPU:", d["env"]["gpu_name"], "| torch", d["env"]["torch"], "| CUDA", d["env"]["cuda_runtime"], "| commit", d["env"]["git_commit"])
for c in json.load(open("results/smoke/checks/smoke_checks.json"))["checks"]:
    print(f"{c['id']:<58}{c['status']}")

selected policy: P1 | accuracy used: False | feasible: True
projected total hours: {'P1': 2.1335969976468716, 'P2': 1.6456591772493816} | ceiling: 3.0
measured penalty % (train step, P1 vs P2): {'A': 32.973659600596065, 'B': 23.812190855335835, 'C': 31.48683455427279}
GPU: Tesla T4 | torch 2.11.0+cu130 | CUDA 13.0 | commit 0541b40a7ba40cdf55fe4faa1749462ac1f0f7ce
V4_P1_same_seed_rerun                                     PASS
V4_P2_same_seed_rerun                                     INFO
V5_shared_init_identical_across_arms_and_processes        PASS
V6_positional_vs_value_embedding_scale                    PASS
V7_demand_floor_keeps_mape_stable                         PASS
V16_inference_memory_excludes_training_state              FAIL
leakage_checks_inside_every_run                           PASS
unit_tests                                                PASS


In [11]:
# 7. Package the smoke outputs (kept separate from any later calibration or main results) and download them.
import glob, subprocess
paths = ["results/smoke", "results/determinism_decision.json", "results/env_report.json", "results/session_manifest.json",
         "requirements.lock.txt"] + glob.glob("results/pip_freeze_*.txt") + glob.glob("data/*.sha256")
paths = [p for p in paths if os.path.exists(p)]
subprocess.run(["zip", "-rq", "week7_smoke_outputs.zip", *paths], check=True)
print("zipped:", paths)
from google.colab import files
files.download("week7_smoke_outputs.zip")

zipped: ['results/smoke', 'results/determinism_decision.json', 'results/env_report.json', 'results/session_manifest.json', 'requirements.lock.txt', 'results/pip_freeze_b506fea2eb.txt', 'data/demand_7c7d3b8c34.sha256']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Stop here.** Send back `week7_smoke_outputs.zip`. Next steps (calibration, pilot, freeze) happen only after the outputs are reviewed. If `requirements.lock.txt` was created in this session, commit it to the repository so later sessions are checked against it.